In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Install & Imports
# ─────────────────────────────────────────────
!pip install -q transformers==4.40.2 accelerate sentencepiece

import torch
import pandas as pd
import re
from tqdm.auto import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

print("All imports done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 89.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
All imports done.


In [3]:
# ─────────────────────────────────────────────
# CELL 2 — Load Phi-3-mini
# ─────────────────────────────────────────────
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          # ← changed from {"": 0} to "auto"
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="eager" # ← add this: disables flash-attn which caused the warning
)
model.eval()

NUM_LAYERS  = model.config.num_hidden_layers   # 32
HIDDEN_SIZE = model.config.hidden_size          # 3072
print(f"num_layers  = {NUM_LAYERS}")
print(f"hidden_size = {HIDDEN_SIZE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

num_layers  = 32
hidden_size = 3072


In [4]:
# ─────────────────────────────────────────────
# CELL 3 — Load your prompt CSV
# ─────────────────────────────────────────────
PROMPT_CSV = "/kaggle/input/datasets/aparnakrishna2407/phi3-experiment/phi3_experiment_final.csv"

prompt_df = pd.read_csv(PROMPT_CSV)
print("Shape:", prompt_df.shape)
print("Columns:")
for c in prompt_df.columns:
    print(" ", c)

Shape: (80, 149)
Columns:
  userId
  gender
  age_group
  itemIds
  artistIds
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-no_info_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-age-group_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-random_userDemo-intersectional_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-intersectional_interact-ICL-Few-shot-1
  prompt_counterfact-Fa

In [5]:
# ─────────────────────────────────────────────
# CELL 4 — Helper: extract Phi-3 hidden states
# ─────────────────────────────────────────────
def get_hidden_states(prompt_text):
    """
    Forward-pass a prompt through Phi-3 with output_hidden_states=True.
    Returns dict {layer_idx (0-indexed): tensor [seq_len, 3072]}
    """
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # outputs.hidden_states is a tuple of length (num_layers + 1)
    # index 0 = embedding layer, 1..32 = transformer layers
    hs_by_layer = {}
    for layer_idx in range(NUM_LAYERS):
        # squeeze batch dim → [seq_len, 3072], move to cpu as float32
        hs_by_layer[layer_idx] = outputs.hidden_states[layer_idx + 1].squeeze(0).cpu().float()

    return hs_by_layer

print("get_hidden_states defined.")

get_hidden_states defined.


In [6]:
test = [col for col in prompt_df.columns if "userDemo-gender" in col and "counterfact-False" in col]
print(len(test), "columns found")
for c in test: print(c)

18 columns found
prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-2
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-2
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1
prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
recommendation_prompt_counterfact-False_sample-random_userDemo-gender_i

In [8]:
# ─────────────────────────────────────────────
# CELL 5 — Build gender-contrastive prompt pairs
# ─────────────────────────────────────────────

GENDER_COL = "prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot"

# Verify the column exists
assert GENDER_COL in prompt_df.columns, f"Column not found: {GENDER_COL}"

def make_male_version(text):
    t = str(text)
    t = re.sub(r'\b[Ff]emale\b', 'male', t)
    t = re.sub(r'\b[Ww]oman\b',  'man',  t)
    t = re.sub(r'\b[Ww]omen\b',  'men',  t)
    t = re.sub(r'\b[Ss]he\b',    'he',   t)
    t = re.sub(r'\b[Hh]er\b',    'his',  t)
    return t

def make_female_version(text):
    t = str(text)
    t = re.sub(r'\b[Mm]ale\b',  'female', t)
    t = re.sub(r'\b[Mm]an\b',   'woman',  t)
    t = re.sub(r'\b[Mm]en\b',   'women',  t)
    t = re.sub(r'\b[Hh]e\b',    'she',    t)
    t = re.sub(r'\b[Hh]is\b',   'her',    t)
    return t

NUM_SV_PAIRS = 30
source_prompts = prompt_df[GENDER_COL].dropna().head(NUM_SV_PAIRS).tolist()
prompt_pairs = [(make_male_version(p), make_female_version(p)) for p in source_prompts]

# Sanity check
print("--- MALE version (first 300 chars) ---")
print(prompt_pairs[0][0][:300])
print("\n--- FEMALE version (first 300 chars) ---")
print(prompt_pairs[0][1][:300])
print(f"\nTotal pairs: {len(prompt_pairs)}")

--- MALE version (first 300 chars) ---
The user is male and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Them

--- FEMALE version (first 300 chars) ---
The user is Female and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Th

Total pairs: 30


In [9]:
# ─────────────────────────────────────────────
# CELL 6 — Compute Phi-3 gender steering vectors
#           (prompt_avg_diff method)
#
# Formula:  V_gender[layer] = mean_over_pairs( h_male[layer].mean(tokens)
#                                             - h_female[layer].mean(tokens) )
#
# This is the same method your professor used for Llama,
# now applied directly to Phi-3's own representation space.
# ─────────────────────────────────────────────
diffs_per_layer = defaultdict(list)   # layer_idx → list of [3072] tensors

print("Extracting hidden states from Phi-3...")
for male_p, female_p in tqdm(prompt_pairs, desc="Prompt pairs"):

    hs_male   = get_hidden_states(male_p)
    hs_female = get_hidden_states(female_p)

    for layer_idx in range(NUM_LAYERS):
        hm = hs_male[layer_idx]    # [seq_len, 3072]
        hf = hs_female[layer_idx]  # [seq_len, 3072]

        # Average over token positions → [3072]
        diff = hm.mean(dim=0) - hf.mean(dim=0)
        diffs_per_layer[layer_idx].append(diff)

# Average across all pairs → one steering vector per layer
phi3_sv_gender = {
    layer_idx: torch.stack(vecs).mean(dim=0)
    for layer_idx, vecs in diffs_per_layer.items()
}

print(f"\nDone. Vectors computed for {len(phi3_sv_gender)} layers.")
print(f"Shape at layer 0: {phi3_sv_gender[0].shape}")  # should be [3072]

# Save the steering vectors
SV_SAVE_PATH = "/kaggle/working/phi3_gender-bias_prompt_avg_diff.pt"
torch.save(phi3_sv_gender, SV_SAVE_PATH)
print(f"Saved steering vectors to: {SV_SAVE_PATH}")

Extracting hidden states from Phi-3...


Prompt pairs:   0%|          | 0/30 [00:00<?, ?it/s]

You are not running the flash-attention implementation, expect numerical differences.
2026-08-09 16:37:20.583271: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786293440.841809      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786293440.925754      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786293441.590483      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786293441.590527      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W


Done. Vectors computed for 32 layers.
Shape at layer 0: torch.Size([3072])
Saved steering vectors to: /kaggle/working/phi3_gender-bias_prompt_avg_diff.pt


In [10]:
# ─────────────────────────────────────────────
# CELL 7 — Define steering hook + generation functions
#
# The formula from your whiteboard:
#   h' = h + λ · V_gender
#
# We apply this at every layer in STEER_LAYERS during generation.
# Hooks are registered before generation and removed immediately after.
# ─────────────────────────────────────────────

# Steer middle layers (most effective for semantic content)
STEER_LAYERS = list(range(8, 24))   # layers 8–23 out of 32

# Lambda values to sweep (negative = steer away from male direction = de-bias)
LAMBDA_VALUES = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]


def _apply_hook(module, inputs, output, steer_vec, lam):
    """
    Forward hook: adds λ·V_norm to hidden states.
    Normalising the vector makes λ a consistent, comparable scale.
    """
    hs = output[0] if isinstance(output, tuple) else output  # [batch, seq_len, 3072]
    v  = steer_vec.to(device=hs.device, dtype=hs.dtype)
    v  = v / (v.norm() + 1e-8)                               # unit vector
    hs = hs + lam * v.view(1, 1, -1)                         # broadcast over batch & seq
    if isinstance(output, tuple):
        return (hs,) + output[1:]
    return hs


def generate_steered(prompt_text, lam, sv_dict=None, max_new_tokens=60):
    """
    Generate with steering hooks active.
    lam=0.0 is equivalent to baseline (no steering effect after normalization,
    but we still call baseline separately for clarity).
    """
    if sv_dict is None:
        sv_dict = phi3_sv_gender

    messages = [{"role": "user", "content": str(prompt_text)}]
    text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(model.device)

    # Register hooks on all steer layers
    handles = []
    for layer_idx in STEER_LAYERS:
        if layer_idx in sv_dict:
            vec = sv_dict[layer_idx]
            h   = model.model.layers[layer_idx].register_forward_hook(
                lambda m, inp, out, v=vec, l=lam: _apply_hook(m, inp, out, v, l)
            )
            handles.append(h)

    try:
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        for h in handles:       # ALWAYS remove hooks
            h.remove()

    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


def generate_baseline(prompt_text, max_new_tokens=60):
    """Generate WITHOUT any steering — pure Phi-3 output."""
    messages = [{"role": "user", "content": str(prompt_text)}]
    text     = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)


print("Hook and generation functions defined.")
print(f"Steering layers: {STEER_LAYERS}")
print(f"Lambda values to sweep: {LAMBDA_VALUES}")

Hook and generation functions defined.
Steering layers: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Lambda values to sweep: [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]


In [11]:
# ─────────────────────────────────────────────
# CELL 8 — Quick sanity check on ONE prompt
#           before running the full experiment
# ─────────────────────────────────────────────
test_prompt = prompt_df[GENDER_COL].iloc[0]
print("TEST PROMPT (first 300 chars):")
print(str(test_prompt)[:300])
print()

print("=== BASELINE ===")
print(generate_baseline(test_prompt))
print()

print("=== STEERED λ = -1.0 (de-biased) ===")
print(generate_steered(test_prompt, lam=-1.0))
print()

print("=== STEERED λ = +1.0 (amplified) ===")
print(generate_steered(test_prompt, lam=1.0))

TEST PROMPT (first 300 chars):
The user is Female and Early Adult (≤24 yrs). The user has listenned to the following songs in the past, organized as (Song - Artist):

- "Gangsta Bop" by Akon
- "I Can'T Wait" by Akon
- "My Love" by Joe
- "One For Me" by Lloyd
- "Get'Cha Head In The Game (Pop Version)" by B5
- "The Backyardigans Th

=== BASELINE ===
Based on the user's musical preferences and recent listening history, the top-3 suitable next recommendations could be:

1. "Bamboo" by Akon - This song by Akon has a similar style and energy as "Gangsta Bop" and "

=== STEERED λ = -1.0 (de-biased) ===
Based on the user's recent listening history and preferences, the top-3 suitable next recommendations could be:

1. "Get'Cha Head In The Game (Pop Version)" by B5 - This song is by the same artist as "U Got Me" and shares

=== STEERED λ = +1.0 (amplified) ===
Based on the user's musical preferences and recent listening history, the top-3 suitable next recommendations could be:

1. "Bam Bam" by 

In [12]:
# ─────────────────────────────────────────────
# CELL 9 — Run experiment with auto-checkpoint
#           (resumes from where it stopped)
# ─────────────────────────────────────────────
import os

OUT_CSV     = "/kaggle/working/phi3_steered_experiment2.csv"
CHECKPOINT  = "/kaggle/working/checkpoint_steered.csv"
DONE_FILE   = "/kaggle/working/completed_cols.txt"
MAIN_LAMBDA = -1.0

# All gender columns (counterfactual-False only)
gender_cols_to_steer = [
    col for col in prompt_df.columns
    if "userDemo-gender" in col and "counterfact-False" in col
]
print(f"Total gender columns to steer: {len(gender_cols_to_steer)}")
for c in gender_cols_to_steer:
    print(" ", c)

# ── Load checkpoint if exists ──
if os.path.exists(CHECKPOINT) and os.path.exists(DONE_FILE):
    print("\nCheckpoint found — resuming...")
    checkpoint_df = pd.read_csv(CHECKPOINT)
    result_cols   = [c for c in checkpoint_df.columns if c not in prompt_df.columns]
    results_df    = checkpoint_df[result_cols].copy()
    results_df.index = prompt_df.index
    with open(DONE_FILE) as f:
        done_cols = [line.strip() for line in f if line.strip()]
    print(f"Already completed: {len(done_cols)}/{len(gender_cols_to_steer)} columns")
else:
    print("\nNo checkpoint — starting fresh...")
    results_df = pd.DataFrame(index=prompt_df.index)
    done_cols  = []

# ── Only run what's not done yet ──
remaining = [c for c in gender_cols_to_steer if c not in done_cols]
print(f"Remaining: {len(remaining)} columns\n")

for col in tqdm(remaining, desc="Gender columns"):
    print(f"\nRunning: {col}")
    baseline_out = []
    steered_out  = []

    for prompt in tqdm(prompt_df[col], desc=col, leave=False):
        try:
            b = generate_baseline(str(prompt))
        except Exception as e:
            print(f"  Baseline error: {e}"); b = ""
        try:
            s = generate_steered(str(prompt), lam=MAIN_LAMBDA)
        except Exception as e:
            print(f"  Steered error: {e}"); s = ""

        baseline_out.append(b)
        steered_out.append(s)

    results_df[f"baseline_{col}"]                 = baseline_out
    results_df[f"steered_lam{MAIN_LAMBDA}_{col}"] = steered_out

    # ── Save checkpoint after every column ──
    pd.concat([prompt_df, results_df], axis=1).to_csv(CHECKPOINT, index=False)
    done_cols.append(col)
    with open(DONE_FILE, "w") as f:
        f.write("\n".join(done_cols))
    print(f"  ✓ Checkpoint saved ({len(done_cols)}/{len(gender_cols_to_steer)} done)")

# ── Final save ──
final_df = pd.concat([prompt_df, results_df], axis=1)
final_df.to_csv(OUT_CSV, index=False)
print(f"\n✓ Final file saved: {OUT_CSV}")
print(f"  Shape: {final_df.shape}")

Total gender columns to steer: 18
  prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-frequent_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-frequent_userDemo-gender_interact-ICL-Few-shot-2
  prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-zero-shot
  prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1
  prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2
  recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot
  recommendation_prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1
  recommendation_prompt_counterf

Gender columns:   0%|          | 0/18 [00:00<?, ?it/s]


Running: prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot


prompt_counterfact-False_sample-random_userDemo-gender_interact-zero-shot:   0%|          | 0/80 [00:00<?, ?it…

  ✓ Checkpoint saved (1/18 done)

Running: prompt_counterfact-False_sample-random_userDemo-gender_interact-ICL-Few-shot-1


...

  ✓ Checkpoint saved (16/18 done)

Running: recommendation_prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1


recommendation_prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-1:   0%| …

  ✓ Checkpoint saved (17/18 done)

Running: recommendation_prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2


recommendation_prompt_counterfact-False_sample-recent-frequent_userDemo-gender_interact-ICL-Few-shot-2:   0%| …

  ✓ Checkpoint saved (18/18 done)

✓ Final file saved: /kaggle/working/phi3_steered_experiment2.csv
  Shape: (80, 185)


In [13]:
# ─────────────────────────────────────────────
# CELL 10 — Evaluation: HR, Gini, Entropy
#            Baseline vs Steered
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

df = pd.read_csv("/kaggle/working/phi3_steered_experiment2.csv")

# Load item catalog for fuzzy matching
all_items = []
for col in df.columns:
    if col.startswith("baseline_") or col.startswith("steered_"):
        all_items.extend(df[col].dropna().tolist())

def parse_recommendation(text):
    """Extract first recommended artist/song from model output."""
    if pd.isna(text) or text == "":
        return None
    lines = str(text).strip().split("\n")
    for line in lines:
        line = line.strip()
        line = re.sub(r"^\d+[\.\)\-\*]\s*", "", line)
        line = re.sub(r'"([^"]+)".*', r'\1', line)
        line = line.strip()
        if len(line) > 2:
            return line.lower()
    return None

def compute_gini(rec_counts, n_items):
    """Gini index — lower is fairer."""
    counts = np.array(sorted(rec_counts.values()))
    counts = np.append(counts, np.zeros(n_items - len(counts)))
    n = len(counts)
    if counts.sum() == 0:
        return 1.0
    idx = np.arange(1, n + 1)
    return (2 * np.sum(idx * counts) - (n + 1) * counts.sum()) / (n * counts.sum())

def compute_entropy(rec_counts):
    """Entropy — higher is more diverse."""
    total = sum(rec_counts.values())
    if total == 0:
        return 0.0
    probs = np.array([v / total for v in rec_counts.values() if v > 0])
    return -np.sum(probs * np.log(probs))

def compute_hr(parsed, ground_truth_col, df):
    """HR@1 — did the recommendation match ground truth."""
    hits = 0
    total = 0
    for i, rec in enumerate(parsed):
        if rec is None:
            continue
        gt = str(df[ground_truth_col].iloc[i]).lower() if ground_truth_col in df.columns else ""
        if gt and SequenceMatcher(None, rec, gt).ratio() > 0.5:
            hits += 1
        total += 1
    return hits / total if total > 0 else 0.0

# ── Find ground truth column ──
gt_col = [c for c in df.columns if "ground_truth" in c.lower() or "target" in c.lower()]
gt_col = gt_col[0] if gt_col else None
print(f"Ground truth column: {gt_col}")

# ── Get all baseline/steered column pairs ──
baseline_cols = [c for c in df.columns if c.startswith("baseline_")]
steered_cols  = [c for c in df.columns if c.startswith("steered_")]
print(f"Baseline columns: {len(baseline_cols)}")
print(f"Steered columns:  {len(steered_cols)}")

# ── Evaluate each column pair ──
import re
results = []

for b_col in baseline_cols:
    condition = b_col.replace("baseline_", "")
    s_col = f"steered_lam-1.0_{condition}"
    if s_col not in df.columns:
        continue

    b_parsed = [parse_recommendation(t) for t in df[b_col]]
    s_parsed = [parse_recommendation(t) for t in df[s_col]]

    # Count item frequencies
    b_counts = {}
    s_counts = {}
    for r in b_parsed:
        if r: b_counts[r] = b_counts.get(r, 0) + 1
    for r in s_parsed:
        if r: s_counts[r] = s_counts.get(r, 0) + 1

    n_items = 5500  # LastFM-1K catalog size

    b_gini    = compute_gini(b_counts, n_items)
    s_gini    = compute_gini(s_counts, n_items)
    b_entropy = compute_entropy(b_counts)
    s_entropy = compute_entropy(s_counts)
    b_cov     = len(b_counts) / n_items
    s_cov     = len(s_counts) / n_items

    # Extract condition parts for display
    parts = condition.split("_")
    sample  = next((p for p in parts if "sample" in p), "?")
    interact = next((p for p in parts if "interact" in p or "shot" in p.lower() or "Few" in p), "?")

    results.append({
        "condition"        : condition[:60],
        "sample"           : sample,
        "ICL"              : interact,
        "Baseline Gini↓"   : round(b_gini, 5),
        "Steered Gini↓"    : round(s_gini, 5),
        "Gini Δ"           : round(s_gini - b_gini, 5),
        "Baseline Entropy↑": round(b_entropy, 3),
        "Steered Entropy↑" : round(s_entropy, 3),
        "Entropy Δ"        : round(s_entropy - b_entropy, 3),
        "Baseline Cov"     : round(b_cov, 5),
        "Steered Cov"      : round(s_cov, 5),
    })

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("RESULTS: Baseline vs Steered (λ=-1.0) — Gender Columns")
print("="*80)
print(results_df.to_string(index=False))

# ── Summary averages ──
print("\n" + "="*80)
print("SUMMARY AVERAGES")
print("="*80)
print(f"Avg Baseline Gini:    {results_df['Baseline Gini↓'].mean():.5f}")
print(f"Avg Steered  Gini:    {results_df['Steered Gini↓'].mean():.5f}")
print(f"Avg Gini Δ (- = better fairness): {results_df['Gini Δ'].mean():.5f}")
print(f"Avg Baseline Entropy: {results_df['Baseline Entropy↑'].mean():.3f}")
print(f"Avg Steered  Entropy: {results_df['Steered Entropy↑'].mean():.3f}")
print(f"Avg Entropy Δ (+ = better diversity): {results_df['Entropy Δ'].mean():.3f}")

results_df.to_csv("/kaggle/working/phi3_steering_evaluation.csv", index=False)
print("\n✓ Evaluation saved to: /kaggle/working/phi3_steering_evaluation.csv")

Ground truth column: None
Baseline columns: 18
Steered columns:  18

RESULTS: Baseline vs Steered (λ=-1.0) — Gender Columns
                                                   condition                 sample                     ICL  Baseline Gini↓  Steered Gini↓   Gini Δ  Baseline Entropy↑  Steered Entropy↑  Entropy Δ  Baseline Cov  Steered Cov
prompt_counterfact-False_sample-random_userDemo-gender_inter          sample-random      interact-zero-shot        -0.99300       -0.99595 -0.00295              1.662             1.361     -0.301       0.00418      0.00236
prompt_counterfact-False_sample-random_userDemo-gender_inter          sample-random interact-ICL-Few-shot-1        -0.98545       -0.98545  0.00000              4.382             4.382      0.000       0.01455      0.01455
prompt_counterfact-False_sample-random_userDemo-gender_inter          sample-random interact-ICL-Few-shot-2        -0.98545       -0.98545  0.00000              4.382             4.382      0.000       0.014